In [21]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T

In [6]:
spark = SparkSession.builder.appName(
    "RDD and UDFs"
).getOrCreate()

sc = spark.sparkContext

# **RDD (Resilient Distributed Dataset)**

![](../images/RDD.png)

An **RDD (Resilient Distributed Dataset)** is the fundamental building block and core abstraction of Apache Spark. Introduced in the original 2012 Spark paper, it represents an immutable, fault-tolerant, distributed collection of objects that can be processed in parallel across a cluster.

---

### **🏛️ The Three Pillars of an RDD**

1. **Resilient (Fault-Tolerant):**
* If a partition of data on a worker node is lost due to a node failure, Spark doesn't need to replicate data across the network (like Hadoop).
* Instead, it reconstructs the lost partition by replaying the **Lineage Graph** (the deterministic sequence of transformations that created it).


2. **Distributed:**
* Data is automatically partitioned and spread across multiple nodes in the cluster, allowing parallel processing.


3. **Dataset:**
* A collection of partitioned data elements (which can be primitive types, tuples, or custom Python/Java/Scala objects).

---

### **⚙️ Core Characteristics & Execution Model**

#### **1. Immutability**

Once created, an RDD cannot be changed. Every transformation applied to an RDD (like `.map()` or `.filter()`) produces a **brand-new RDD**, preserving lineage and making fault-tolerance trivial.

#### **2. Lazy Evaluation**

Transformations on RDDs do not compute results immediately. Spark builds a **Directed Acyclic Graph (DAG)** of transformations and only triggers physical computation when an **Action** (e.g., `.collect()`, `.count()`, `.saveAsTextFile()`) is called.

* **Transformations (Lazy):** `map`, `filter`, `flatMap`, `distinct` (returns a new RDD).
* **Actions (Eager):** `collect`, `take`, `reduce`, `count`, `save` (triggers cluster execution).

---

### **🔄 RDDs vs. DataFrames & Datasets**

While modern PySpark code almost exclusively uses **DataFrames** and **Datasets**, RDDs still operate silently underneath every DataFrame.

| Feature | **RDD** | **DataFrame / Dataset** |
| --- | --- | --- |
| **API Style** | Low-level functional programming (lambdas, closures). | High-level relational API (`select`, `groupby`, SQL expressions). |
| **Optimization** | **None.** Spark executes your code exactly as written (black-box closures). | **Catalyst Optimizer & Tungsten.** Automatically optimizes query plans, predicate pushdown, and binary memory layouts. |
| **Type Safety** | Compile-time/runtime object checking depending on language. | Schema-enforced tabular structure. |
| **When to use** | Unstructured binary data, custom low-level partitioning control, or legacy codebases. | 99% of modern big data ETL, analytics, and machine learning pipelines. |

### **💡 Example: Low-Level RDD Usage in PySpark**

Even in a DataFrame workflow, you can drop down to the underlying RDD layer using `.rdd`:

```python
# Convert a DataFrame to an RDD of Row objects
rdd = full_log.rdd

# Apply low-level Python transformations using a lambda function
filtered_rdd = rdd.filter(lambda row: row["duration_seconds"] > 3600)

# Trigger an action
print(filtered_rdd.count())
```

## **Map Operation**

In [5]:
collection = [1, "two", 3.0, ("four", 4), {"five": 5}]

In [7]:
collection_rdd = sc.parallelize(collection)

collection_rdd

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:326

In [8]:
from py4j.protocol import Py4JJavaError

In [9]:
def add_one(value):
    return value + 1

In [11]:
collection_rdd = collection_rdd.map(add_one)

try:
    print(collection_rdd.collect())
except Py4JJavaError:
    pass

26/11/14 17:16:01 ERROR Executor: Exception in task 1.0 in stage 0.0 (TID 1)/ 4]
org.apache.spark.api.python.PythonException: [PYTHON_EXCEPTION] An exception was thrown from the Python worker: Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 3622, in main
    process()
  File "/usr/local/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 3612, in process
    serializer.dump_stream(out_iter, outfile)
  File "/usr/local/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/serializers.py", line 257, in dump_stream
    vs = list(itertools.islice(iterator, batch))
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/util.py", line 159, in wrapper
    return f(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1717/3587218469.py", line 2, in add_one
TypeE

PythonException: An exception was thrown from the Python worker:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 3622, in main
    process()
  File "/usr/local/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 3612, in process
    serializer.dump_stream(out_iter, outfile)
  File "/usr/local/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/serializers.py", line 257, in dump_stream
    vs = list(itertools.islice(iterator, batch))
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/util.py", line 159, in wrapper
    return f(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1717/3587218469.py", line 2, in add_one
TypeError: can only concatenate tuple (not "int") to tuple

![](../images/RDD_map.png)

In [12]:
collection_rdd = sc.parallelize(collection)

def safe_add_one(value):
    try:
        return value + 1
    except TypeError:
        return value

In [14]:
collection_rdd = collection_rdd.map(safe_add_one)

try:
    print(collection_rdd.collect())
except Py4JJavaError:
    pass

[2, 'two', 4.0, ('four', 4), {'five': 5}]


## **Filter Operation**

In [15]:
collection_rdd = sc.parallelize(collection)

collection_rdd = collection_rdd.filter(
    lambda elm: isinstance(elm, (float, int))
)

print(collection_rdd.collect())

[1, 3.0]


## **Reduce Operation**

In [16]:
from operator import add

In [17]:
collection_rdd = sc.parallelize([4, 7, 9, 1, 3])

print(collection_rdd.reduce(add))

24


## **DataFrame is an RDD**

In [18]:
df = spark.createDataFrame([[1], [2], [3]], schema=["column"])

df.show()

+------+
|column|
+------+
|     1|
|     2|
|     3|
+------+



In [19]:
df.rdd

MapPartitionsRDD[19] at javaToPython at DirectMethodHandleAccessor.java:103

In [20]:
df.rdd.collect()

[Row(column=1), Row(column=2), Row(column=3)]

# **UDFs (User-Defined Function)**

In Apache Spark, a **UDF (User-Defined Function)** allows you to write custom transformation logic in Python (or Scala/Java) and apply it row-by-row across a distributed DataFrame.

While UDFs provide immense flexibility when built-in SQL functions aren't enough, they are also **one of the most notorious performance bottlenecks** in PySpark if used incorrectly.

---

## **⚙️ How Standard PySpark UDFs Work (And Why They're Slow)**

Spark is written in Scala/Java and runs on the JVM (Java Virtual Machine), leveraging Project Tungsten for lightning-fast, off-heap memory processing.

When you use a standard Python UDF:

1. **Serialization Tax:** Spark must serialize rows from the JVM into Python objects.
2. **Context Switching:** Data is sent across a local socket connection (via Py4J) to a separate Python worker process.
3. **Row-by-Row Execution:** Python evaluates the function on each row individually (interpreting bytecode, incurring massive Python interpreter overhead).
4. **Deserialization Back:** The result is serialized back into JVM format.

Furthermore, standard UDFs act as a **black box** to the Catalyst Optimizer. Spark cannot optimize, reorder, or push down filters through a standard Python UDF because it has no visibility into what the Python code actually does.

---

## **💻 1. Standard PySpark UDF Example**

Here is how you define and register a standard Python UDF using `pyspark.sql.functions.udf`:

```python
import pyspark.sql.functions as F
from pyspark.sql.types import StringType


# 1. Define your custom Python function
def categorize_capacity(capacity_gb):
    if capacity_gb is None:
        return "Unknown"
    elif capacity_gb < 2000:
        return "Small"
    elif capacity_gb < 8000:
        return "Medium"
    else:
        print(f"Large Drive Detected: {capacity_gb} GB")
        return "Large"


# 2. Register the UDF with an explicit return type (crucial for Spark's Catalyst compiler)
categorize_udf = F.udf(categorize_capacity, StringType())

# 3. Apply the UDF in a DataFrame transformation
df_categorized = full_data.select(
    "model", "capacity_GB", categorize_udf("capacity_GB").alias("size_category")
)

```

---

## **🚀 2. The High-Performance Alternative: Pandas UDFs (Vectorized UDFs)**

To solve the row-by-row overhead of standard UDFs, Spark introduced **Pandas UDFs** (powered by **Apache Arrow**).

Instead of passing data row-by-row, Pandas UDFs transfer data in vectorized batches using Apache Arrow's in-memory columnar format. This allows Python to execute operations using vectorized NumPy/Pandas code, achieving speedups of **10x to 100x**.

### **Example: Pandas UDF**

```python
from pyspark.sql.functions import pandas_udf
import pandas as pd


# Define a Pandas UDF taking a Pandas Series and returning a Pandas Series
@pandas_udf("string")
def categorize_pandas_udf(capacity_series: pd.Series) -> pd.Series:
    # Vectorized operations using pandas/numpy logic (no row-by-row python loops)
    return pd.select(
        [
            capacity_series < 2000,
            (capacity_series >= 2000) & (capacity_series < 8000),
            capacity_series >= 8000,
        ],
        ["Small", "Medium", "Large"],
        default="Unknown",
    )


# Apply it just like a regular column expression
df_vectorized = full_data.select(
    "model",
    "capacity_GB",
    categorize_pandas_udf("capacity_GB").alias("size_category"),
)

```

---

## **📋 Best Practices & Engineering Guidelines for UDFs**

1. **The #1 Rule of UDFs:** *Don't use them if a built-in Spark SQL function exists.*
* Built-in functions (`F.col`, `F.when`, `F.regexp_extract`, `F.coalesce`, mathematical expressions) are written in optimized Java/Scala, execute directly on Tungsten binary memory, and allow full Catalyst optimization.


2. **Always Declare Return Types:** In standard UDFs, always specify the return type (`StringType()`, `LongType()`, etc.). Letting Spark guess can lead to runtime crashes or type mismatch errors.
3. **Use Pandas UDFs for Heavy Python Libraries:** If you need to run machine learning model inference (e.g., PyTorch, Scikit-Learn), cryptographic hashing, or complex statistical routines via Pandas/NumPy, always use `@pandas_udf` to leverage Apache Arrow vectorization.
4. **Prefer SQL Expressions (`selectExpr`):** For simple arithmetic, conditional logic (`CASE WHEN`), or string manipulations, `selectExpr()` or standard expressions are cleaner, faster, and easier for the optimizer to handle than Python UDFs.

In [31]:
fractions = [[x, y] for x in range(100) for y in range(1, 100)]

frac_df = spark.createDataFrame(
    fractions, ["numerator", "denominator"]
)

In [32]:
frac_df = frac_df.select(
    F.array(F.col('numerator'), F.col('denominator')).alias('fraction')
)

frac_df.show(5)

+--------+
|fraction|
+--------+
|  [0, 1]|
|  [0, 2]|
|  [0, 3]|
|  [0, 4]|
|  [0, 5]|
+--------+
only showing top 5 rows


In [24]:
from fractions import Fraction
from typing import Tuple, Optional

In [25]:
Frac = Tuple[int, int]

In [26]:
def py_reduce_fraction(frac: Frac) -> Optional[Frac]:
    """Reduce a fraction represented as a 2-tuple of integers."""

    num, denom = frac
    if denom:
        answer = Fraction(num, denom)
        return answer.numerator, answer.denominator
    return None

In [27]:
assert py_reduce_fraction((3, 6)) == (1, 2)
assert py_reduce_fraction((1, 0)) is None

In [28]:
def py_fraction_to_float(frac: Frac) -> Optional[float]:
    """Transforms a fraction represented as a 2-tuple of integers into a float."""

    num, denom = frac
    if denom:
        return num / denom
    return None

In [29]:
assert py_fraction_to_float((2, 8)) == 0.25
assert py_fraction_to_float((10, 0)) is None

In [33]:
sparkFrac = T.ArrayType(T.LongType())

reduce_fraction = F.udf(py_reduce_fraction, sparkFrac)

frac_df = frac_df.withColumn(
    'reduced_fraction', reduce_fraction(F.col('fraction'))
)

frac_df.show(5, False)

+--------+----------------+
|fraction|reduced_fraction|
+--------+----------------+
|[0, 1]  |[0, 1]          |
|[0, 2]  |[0, 1]          |
|[0, 3]  |[0, 1]          |
|[0, 4]  |[0, 1]          |
|[0, 5]  |[0, 1]          |
+--------+----------------+
only showing top 5 rows


In [34]:
@F.udf(T.DoubleType())
def fraction_to_float(frac: Frac) -> Optional[float]:
    """Transforms a fraction represented as a 2-tuple of integers into a float."""

    num, denom = frac
    if denom:
        return num / denom
    
    return None

In [35]:
frac_df = frac_df.withColumn(
    "fraction_float", fraction_to_float(F.col("reduced_fraction"))
)

frac_df.select("reduced_fraction", "fraction_float").distinct().show(5, False)

[Stage 12:>                                                         (0 + 4) / 4]

+----------------+-------------------+
|reduced_fraction|fraction_float     |
+----------------+-------------------+
|[3, 50]         |0.06               |
|[3, 67]         |0.04477611940298507|
|[7, 76]         |0.09210526315789473|
|[9, 23]         |0.391304347826087  |
|[9, 25]         |0.36               |
+----------------+-------------------+
only showing top 5 rows
